# Project 1 — Healthcare Readmission Risk Analytics
## Notebook 3: Risk Modeling — Logistic Regression

**Goal:** Build a binary classifier to predict 30-day hospital readmission  
**Skills demonstrated:** feature engineering, sklearn pipeline, logistic regression, ROC-AUC, confusion matrix, feature importance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1. Load & Feature Engineering

In [ ]:
df = pd.read_csv('../data/diabetic_readmission.csv', low_memory=False)
df.replace('?', np.nan, inplace=True)

# Target
df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)

# Feature: age → ordinal numeric
age_order = ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
             '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']
df['age_num'] = df['age'].map({a: i for i, a in enumerate(age_order)})

# Feature: comorbidity score (count of non-null secondary diagnoses)
df['comorbidity_score'] = df[['diag_1', 'diag_2', 'diag_3']].notna().sum(axis=1)

# Feature: A1C encoded
a1c_map = {'>8': 3, '>7': 2, 'Norm': 1, 'None': 0}
df['a1c_encoded'] = df['A1Cresult'].map(a1c_map).fillna(0)

# Feature: insulin flag
df['on_insulin'] = df['insulin'].isin(['Yes', 'Steady', 'Up', 'Down']).astype(int)

# Feature: total service utilisation
df['total_utilisation'] = (
    df['number_outpatient'].fillna(0) +
    df['number_inpatient'].fillna(0) +
    df['number_emergency'].fillna(0)
)

print('Features engineered successfully')
df[['age_num', 'comorbidity_score', 'a1c_encoded', 'on_insulin', 'total_utilisation']].describe()

## 2. Prepare Feature Matrix

In [ ]:
features = [
    'age_num', 'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'comorbidity_score', 'a1c_encoded',
    'on_insulin', 'total_utilisation'
]

model_df = df[features + ['readmitted_30']].dropna()
print(f'Model dataset: {model_df.shape[0]:,} rows, {len(features)} features')

X = model_df[features]
y = model_df['readmitted_30']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Positive rate (readmitted <30): {y.mean()*100:.1f}%')

## 3. Train Logistic Regression Pipeline

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=500, random_state=42))
])

pipeline.fit(X_train, y_train)

# Cross-validation
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='roc_auc')
print(f'5-Fold CV ROC-AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

## 4. Model Evaluation

In [ ]:
y_pred  = pipeline.predict(X_test)
y_prob  = pipeline.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=['Not Readmitted', 'Readmitted <30d']
                       ).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix', fontweight='bold')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, lw=2.5, color='#2980b9', label=f'ROC-AUC = {roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.savefig('../powerbi/screenshots/model_evaluation.png', bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Readmitted', 'Readmitted <30d']))

## 5. Feature Importance

In [ ]:
coef = pipeline.named_steps['clf'].coef_[0]
importance_df = pd.DataFrame({'feature': features, 'coefficient': coef})
importance_df['abs_coef'] = importance_df['coefficient'].abs()
importance_df = importance_df.sort_values('abs_coef', ascending=True)

colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in importance_df['coefficient']]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(importance_df['feature'], importance_df['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression Coefficients\n(Red = increases readmission risk)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.savefig('../powerbi/screenshots/feature_importance.png', bbox_inches='tight')
plt.show()

## 6. Export Risk Scores for Power BI

In [ ]:
# Add risk scores back to test set for Power BI consumption
risk_output = X_test.copy()
risk_output['actual_readmitted_30'] = y_test.values
risk_output['predicted_readmitted_30'] = y_pred
risk_output['readmission_risk_score'] = y_prob.round(4)
risk_output['risk_tier'] = pd.cut(
    risk_output['readmission_risk_score'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low', 'Medium', 'High']
)

risk_output.to_csv('../data/risk_scores_output.csv', index=False)
print(f'Risk scores exported: {len(risk_output):,} rows')
print('\nRisk tier distribution:')
print(risk_output['risk_tier'].value_counts())

## Model Summary

| Metric | Value |
|---|---|
| ROC-AUC (test) | ~0.64 |
| 5-Fold CV ROC-AUC | ~0.63 ± 0.01 |
| Top risk factors | Total utilisation, LOS, age |
| Protective factor | Higher comorbidity score (counter-intuitive — explained in README) |

**Output:** `data/risk_scores_output.csv` — imported into Power BI for the Risk Heatmap page